# Baby Step 5 — Execute Controlled Diligence and Refresh Evidence

**Project:** Corporate Civil Litigation Exo-Brain  
**Author:** Alejandro Reynoso  
**Persistent vault:** `/content/drive/MyDrive/Alejandro-Reynoso-Corporate-Civil-Litigation-ExoBrain`

## Objective

Baby Step 5 performs the controlled internal diligence authorized by DEC-004.

For each of the five active matters, the notebook:

1. defines bounded diligence questions;
2. generates synthetic diligence evidence;
3. creates new source and claim records;
4. resolves, narrows, or preserves contradictions;
5. recalculates claim confidence;
6. refreshes matter permission states;
7. produces a diligence readout;
8. records DEC-005;
9. preserves Recommendation V1;
10. still prohibits Recommendation V2.

The purpose is not to accumulate more documents. It is to convert open questions into governed evidence and determine whether reliance can widen.


## Diligence workstreams

Each matter receives four controlled workstreams:

- legal and contractual;
- factual and operational;
- financial and damages;
- procedural and remedy.

The workstreams differ by matter type. They are designed to answer the questions that previously constrained reliance.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
import csv
import datetime
import statistics
from collections import defaultdict, Counter

VAULT = Path(r"/content/drive/MyDrive/Alejandro-Reynoso-Corporate-Civil-Litigation-ExoBrain")

if not VAULT.exists():
    raise FileNotFoundError("Run Baby Step 0 first.")

state_path = VAULT / "00_System" / "Workflow_State.json"
state = json.loads(state_path.read_text(encoding="utf-8"))

if 4 not in state.get("completed_steps", []):
    raise RuntimeError("Baby Step 4 is not complete.")

matters = json.loads(
    (VAULT / "data" / "active_matters.json").read_text(encoding="utf-8")
)

recommendations = json.loads(
    (VAULT / "data" / "baby_step_1_recommendations_v1.json").read_text(encoding="utf-8")
)

sources = json.loads(
    (VAULT / "data" / "baby_step_3_sources.json").read_text(encoding="utf-8")
)

claims = json.loads(
    (VAULT / "data" / "baby_step_3_claims.json").read_text(encoding="utf-8")
)

contradictions = json.loads(
    (VAULT / "data" / "baby_step_3_contradictions.json").read_text(encoding="utf-8")
)

committee_paths = json.loads(
    (VAULT / "data" / "baby_step_4_committee_paths.json").read_text(encoding="utf-8")
)

print("Matters:", len(matters))
print("Existing sources:", len(sources))
print("Existing claims:", len(claims))
print("Open contradictions:", len(contradictions))


## Matter-specific diligence plans

The notebook defines concrete questions rather than generic requests for “more information.”

Examples include:

- whether a contractual trigger was satisfied;
- whether expert determination is exclusive;
- whether technical deployment exceeded license scope;
- whether deadlock procedures were exhausted;
- whether damages were caused and mitigated.


In [ ]:
DILIGENCE_PLANS = {
    "MAT-001": {
        "legal_contractual": [
            "Confirm transfer-restriction trigger language",
            "Determine whether consent was contractually discretionary",
            "Test waiver and prior-course-of-dealing defenses"
        ],
        "factual_operational": [
            "Reconstruct transfer-notice chronology",
            "Compare board minutes with email correspondence"
        ],
        "financial_damages": [
            "Estimate value impairment under transfer scenarios"
        ],
        "procedural_remedy": [
            "Test irreparable-harm evidence",
            "Assess practical availability of interim relief"
        ]
    },
    "MAT-002": {
        "legal_contractual": [
            "Confirm scope of expert-determination clause",
            "Test contractual time bars and jurisdiction"
        ],
        "factual_operational": [
            "Reconcile closing statement and accounting workpapers",
            "Identify consistency of accounting policies"
        ],
        "financial_damages": [
            "Recalculate working-capital adjustment",
            "Segment disputed deferred-revenue treatment"
        ],
        "procedural_remedy": [
            "Map court-versus-expert sequencing"
        ]
    },
    "MAT-003": {
        "legal_contractual": [
            "Map license scope and deployment restrictions",
            "Review confidentiality and audit provisions"
        ],
        "factual_operational": [
            "Compare deployment logs with licensed environments",
            "Trace access to confidential technical materials"
        ],
        "financial_damages": [
            "Recalculate royalty exposure and limitation clauses"
        ],
        "procedural_remedy": [
            "Assess technical-proof requirements for targeted relief"
        ]
    },
    "MAT-004": {
        "legal_contractual": [
            "Confirm deadlock definition and escalation requirements",
            "Review buy-sell and dissolution mechanisms"
        ],
        "factual_operational": [
            "Reconstruct committee voting history",
            "Test whether governance paralysis is continuing"
        ],
        "financial_damages": [
            "Refresh buyout and dissolution-value scenarios"
        ],
        "procedural_remedy": [
            "Assess threshold for equitable relief"
        ]
    },
    "MAT-005": {
        "legal_contractual": [
            "Interpret force-majeure and damages limitations",
            "Test notice and mitigation obligations"
        ],
        "factual_operational": [
            "Reconstruct production and shipping interruption",
            "Identify alternative supply options"
        ],
        "financial_damages": [
            "Separate direct, consequential, and avoidable losses"
        ],
        "procedural_remedy": [
            "Assess partial-summary-judgment issues"
        ]
    }
}

assert set(DILIGENCE_PLANS) == {m["matter_id"] for m in matters}

(VAULT / "data" / "baby_step_5_diligence_plans.json").write_text(
    json.dumps(DILIGENCE_PLANS, indent=2),
    encoding="utf-8"
)

print("Diligence plans created:", len(DILIGENCE_PLANS))


## Synthetic diligence evidence

Each workstream produces bounded synthetic findings.

Every new finding becomes:

- a new source record;
- one or more atomic claims;
- a definition or limitation;
- a link to the contradiction or recommendation affected.


In [ ]:
new_sources = []
new_claims = []

def add_source(source_id, matter_id, source_type, origin, summary,
               authority, independence, directness, limitations):
    new_sources.append({
        "source_id": source_id,
        "matter_id": matter_id,
        "source_type": source_type,
        "origin": origin,
        "source_date": datetime.date.today().isoformat(),
        "authority_score": authority,
        "independence_score": independence,
        "recency_score": 100,
        "directness_score": directness,
        "summary": summary,
        "limitations": limitations,
        "synthetic": True
    })

def add_claim(claim_id, matter_id, claim_type, text, source_ids,
              dependencies, confidence, definition, limitations):
    new_claims.append({
        "claim_id": claim_id,
        "matter_id": matter_id,
        "claim_type": claim_type,
        "claim_text": text,
        "supporting_source_ids": source_ids,
        "decision_dependencies": dependencies,
        "confidence_score": confidence,
        "definition": definition,
        "limitations": limitations,
        "synthetic": True
    })

FINDINGS = {
    "MAT-001": [
        {
            "source_type": "Contract Analysis",
            "origin": "Transfer restriction clause review",
            "summary": "The synthetic clause requires consent after a defined transfer notice and preserves equitable relief.",
            "authority": 90, "independence": 72, "directness": 95,
            "claim": "The transfer restriction was contractually triggered by the synthetic notice.",
            "claim_type": "Reported Fact", "confidence": 84,
            "definition": "Trigger assessed against the synthetic agreement language.",
            "limitations": "No real contract or jurisdictional rule."
        },
        {
            "source_type": "Chronology Reconstruction",
            "origin": "Transfer notice and board record reconciliation",
            "summary": "The notice preceded the disputed consent decision and the chronology is internally consistent.",
            "authority": 80, "independence": 76, "directness": 88,
            "claim": "The synthetic notice chronology supports a timely request for relief.",
            "claim_type": "Analyst Inference", "confidence": 73,
            "definition": "Timeliness measured from synthetic event dates.",
            "limitations": "Inference remains subject to procedural confirmation."
        }
    ],
    "MAT-002": [
        {
            "source_type": "Accounting Reconciliation",
            "origin": "Closing statement and workpaper reconciliation",
            "summary": "The disputed adjustment is primarily driven by deferred-revenue classification.",
            "authority": 88, "independence": 80, "directness": 94,
            "claim": "Deferred-revenue classification explains most of the synthetic adjustment variance.",
            "claim_type": "Derived Calculation", "confidence": 86,
            "definition": "Variance measured against the synthetic closing statement.",
            "limitations": "Not an audit opinion."
        },
        {
            "source_type": "Contract Analysis",
            "origin": "Expert-determination clause review",
            "summary": "The clause assigns accounting disputes to expert determination but reserves pure contract interpretation.",
            "authority": 92, "independence": 75, "directness": 96,
            "claim": "The synthetic dispute should be segmented between expert accounting issues and judicial contract interpretation.",
            "claim_type": "Legal Proposition", "confidence": 82,
            "definition": "Segmentation based on synthetic clause wording.",
            "limitations": "No real governing law."
        }
    ],
    "MAT-003": [
        {
            "source_type": "Technical Reconciliation",
            "origin": "Deployment log and license-scope mapping",
            "summary": "A subset of deployments exceeds the synthetic licensed environment.",
            "authority": 84, "independence": 82, "directness": 94,
            "claim": "Some synthetic deployments fall outside the documented license scope.",
            "claim_type": "Reported Fact", "confidence": 85,
            "definition": "Out-of-scope deployment measured against synthetic environment definitions.",
            "limitations": "No forensic image or real system."
        },
        {
            "source_type": "Royalty Model",
            "origin": "Revised royalty and limitation analysis",
            "summary": "The plausible royalty range is narrower than the original modeled exposure.",
            "authority": 78, "independence": 68, "directness": 86,
            "claim": "The synthetic royalty exposure is lower than the original broad estimate.",
            "claim_type": "Estimate", "confidence": 76,
            "definition": "Range uses synthetic deployment counts and contract rates.",
            "limitations": "Not a damages expert opinion."
        }
    ],
    "MAT-004": [
        {
            "source_type": "Governance Chronology",
            "origin": "Committee voting and escalation reconstruction",
            "summary": "The synthetic escalation process was substantially completed without resolution.",
            "authority": 86, "independence": 74, "directness": 92,
            "claim": "The contractual deadlock process was substantially exhausted.",
            "claim_type": "Reported Fact", "confidence": 83,
            "definition": "Exhaustion assessed against synthetic governance milestones.",
            "limitations": "No real corporate record."
        },
        {
            "source_type": "Valuation Scenario",
            "origin": "Updated buyout and dissolution analysis",
            "summary": "A staged buyout preserves more synthetic enterprise value than immediate dissolution.",
            "authority": 72, "independence": 62, "directness": 80,
            "claim": "A negotiated staged buyout is economically superior to immediate synthetic dissolution.",
            "claim_type": "Analyst Inference", "confidence": 70,
            "definition": "Comparison uses synthetic enterprise-value scenarios.",
            "limitations": "Model-dependent and not a valuation opinion."
        }
    ],
    "MAT-005": [
        {
            "source_type": "Causation Reconstruction",
            "origin": "Production, shipping, and customer-loss reconciliation",
            "summary": "Only part of the claimed downstream loss is directly linked to the synthetic interruption.",
            "authority": 83, "independence": 78, "directness": 90,
            "claim": "The synthetic causation record supports a narrower direct-loss range.",
            "claim_type": "Derived Calculation", "confidence": 82,
            "definition": "Direct loss excludes unsupported downstream assumptions.",
            "limitations": "Not an expert causation opinion."
        },
        {
            "source_type": "Mitigation Review",
            "origin": "Alternative supply analysis",
            "summary": "Alternative supply was partially available but more costly and delayed.",
            "authority": 76, "independence": 74, "directness": 84,
            "claim": "The claimant had partial but imperfect synthetic mitigation options.",
            "claim_type": "Reported Fact", "confidence": 79,
            "definition": "Availability measured during the synthetic interruption window.",
            "limitations": "No real market evidence."
        }
    ]
}

for matter_id, findings in FINDINGS.items():
    recommendation = next(
        r for r in recommendations
        if r["matter_id"] == matter_id
    )

    for idx, finding in enumerate(findings, start=1):
        sid = f"SRC-{matter_id}-DIL-{idx:02d}"
        cid = f"CLM-{matter_id}-DIL-{idx:02d}"

        add_source(
            sid,
            matter_id,
            finding["source_type"],
            finding["origin"],
            finding["summary"],
            finding["authority"],
            finding["independence"],
            finding["directness"],
            finding["limitations"]
        )

        add_claim(
            cid,
            matter_id,
            finding["claim_type"],
            finding["claim"],
            [sid],
            [recommendation["recommendation_id"], "DEC-004"],
            finding["confidence"],
            finding["definition"],
            finding["limitations"]
        )

print("New diligence sources:", len(new_sources))
print("New diligence claims:", len(new_claims))


## Contradiction resolution engine

The notebook does not delete contradictions.

Each contradiction is assigned one of four refreshed states:

- RESOLVED;
- NARROWED;
- REMAINS OPEN;
- SUPERSEDED BY BETTER DEFINITION.

The original contradiction record remains part of history.


In [ ]:
updated_contradictions = []

for contradiction in contradictions:
    updated = dict(contradiction)
    mid = contradiction["matter_id"]
    category = contradiction["category"]

    if category == "Assumption Dependency":
        updated["refreshed_status"] = "RESOLVED"
        updated["resolution_note"] = (
            "Controlled diligence confirmed the synthetic procedural posture."
        )
        updated["resolution_source_ids"] = [
            f"SRC-{mid}-DIL-01"
        ]

    elif category == "Evidence Sufficiency":
        updated["refreshed_status"] = "NARROWED"
        updated["resolution_note"] = (
            "New diligence improved support but did not create a final expert-grade conclusion."
        )
        updated["resolution_source_ids"] = [
            f"SRC-{mid}-DIL-02"
        ]

    elif category == "Strategy Conflict":
        updated["refreshed_status"] = "REMAINS OPEN"
        updated["resolution_note"] = (
            "Diligence improved evidence but a recommendation change requires a later governed version."
        )
        updated["resolution_source_ids"] = [
            f"SRC-{mid}-DIL-01",
            f"SRC-{mid}-DIL-02"
        ]

    elif category == "Authority Treatment":
        updated["refreshed_status"] = "NARROWED"
        updated["resolution_note"] = (
            "Matter-specific evidence narrows factual uncertainty, but legal-authority treatment remains relevant."
        )
        updated["resolution_source_ids"] = [
            f"SRC-{mid}-DIL-01"
        ]

    else:
        updated["refreshed_status"] = "REMAINS OPEN"
        updated["resolution_note"] = "No sufficient diligence result."
        updated["resolution_source_ids"] = []

    updated["refreshed_at"] = datetime.datetime.now().isoformat()
    updated_contradictions.append(updated)

(VAULT / "data" / "baby_step_5_refreshed_contradictions.json").write_text(
    json.dumps(updated_contradictions, indent=2),
    encoding="utf-8"
)

status_counts = Counter(
    item["refreshed_status"]
    for item in updated_contradictions
)

print("Contradiction refresh:", dict(status_counts))


## Confidence and permission refresh

New diligence claims are added to the existing evidence base.

Matter-level confidence is refreshed using:

- original claim confidence;
- new diligence confidence;
- contradiction-resolution status;
- remaining high-severity issues.

Recommendation V1 remains unchanged.


In [ ]:
all_sources = sources + new_sources
all_claims = claims + new_claims

claims_by_matter = defaultdict(list)
for claim in all_claims:
    claims_by_matter[claim["matter_id"]].append(claim)

refreshed_permissions = []

for matter in matters:
    mid = matter["matter_id"]
    local_claims = claims_by_matter[mid]
    local_cons = [
        c for c in updated_contradictions
        if c["matter_id"] == mid
    ]

    avg_conf = round(
        statistics.mean(
            c["confidence_score"]
            for c in local_claims
        ),
        2
    )

    unresolved_high = [
        c for c in local_cons
        if c["severity"] == "HIGH"
        and c["refreshed_status"] == "REMAINS OPEN"
    ]

    narrowed = [
        c for c in local_cons
        if c["refreshed_status"] == "NARROWED"
    ]

    if unresolved_high:
        permission = "QUALIFIED_INTERNAL_USE"
    elif narrowed:
        permission = "INTERNAL_USE_WITH_DISCLOSURE"
    else:
        permission = "INTERNAL_USE"

    refreshed_permissions.append({
        "matter_id": mid,
        "refreshed_average_claim_confidence": avg_conf,
        "remaining_high_severity_contradictions": len(unresolved_high),
        "narrowed_contradictions": len(narrowed),
        "permission_state": permission,
        "recommendation_v1_preserved": True,
        "recommendation_v2_allowed": False,
        "external_action_allowed": False
    })

(VAULT / "data" / "baby_step_5_refreshed_permission_state.json").write_text(
    json.dumps(refreshed_permissions, indent=2),
    encoding="utf-8"
)

print(json.dumps(refreshed_permissions, indent=2))


## Write new governed objects

The notebook adds diligence sources and claims to the same vault, while preserving the original Baby Step 3 records.


In [ ]:
def write_note(path, lines):
    path.write_text(
        "\n".join(lines).strip() + "\n",
        encoding="utf-8"
    )

source_dir = VAULT / "14_Sources"
claim_dir = VAULT / "15_Claims"
diligence_dir = VAULT / "17_Diligence"
diligence_dir.mkdir(parents=True, exist_ok=True)

for source in new_sources:
    lines = [
        "---",
        f"source_id: {source['source_id']}",
        f"matter_id: {source['matter_id']}",
        f"source_type: \"{source['source_type']}\"",
        "baby_step: 5",
        "synthetic: true",
        "---",
        "",
        f"# {source['source_id']}",
        "",
        f"- Matter: [[../02_Active_Matters/{source['matter_id']}]]",
        f"- Origin: {source['origin']}",
        f"- Authority: {source['authority_score']}/100",
        f"- Independence: {source['independence_score']}/100",
        f"- Directness: {source['directness_score']}/100",
        "",
        "## Summary",
        "",
        source["summary"],
        "",
        "## Limitations",
        "",
        source["limitations"]
    ]
    write_note(
        source_dir / f"{source['source_id']}.md",
        lines
    )

for claim in new_claims:
    lines = [
        "---",
        f"claim_id: {claim['claim_id']}",
        f"matter_id: {claim['matter_id']}",
        f"claim_type: \"{claim['claim_type']}\"",
        f"confidence_score: {claim['confidence_score']}",
        "baby_step: 5",
        "synthetic: true",
        "---",
        "",
        f"# {claim['claim_id']}",
        "",
        "## Claim",
        "",
        claim["claim_text"],
        "",
        "## Supporting source",
        ""
    ]
    lines += [
        f"- [[../14_Sources/{sid}]]"
        for sid in claim["supporting_source_ids"]
    ]
    lines += [
        "",
        "## Definition",
        "",
        claim["definition"],
        "",
        "## Limitations",
        "",
        claim["limitations"],
        "",
        "## Decision dependencies",
        ""
    ]
    lines += [
        f"- {dep}"
        for dep in claim["decision_dependencies"]
    ]

    write_note(
        claim_dir / f"{claim['claim_id']}.md",
        lines
    )

for matter in matters:
    mid = matter["matter_id"]
    plan = DILIGENCE_PLANS[mid]
    local_sources = [
        s for s in new_sources
        if s["matter_id"] == mid
    ]
    local_claims = [
        c for c in new_claims
        if c["matter_id"] == mid
    ]
    local_cons = [
        c for c in updated_contradictions
        if c["matter_id"] == mid
    ]
    permission = next(
        p for p in refreshed_permissions
        if p["matter_id"] == mid
    )

    lines = [
        f"# {mid} — Controlled Diligence Record",
        "",
        "## Workstreams",
        ""
    ]

    for workstream, questions in plan.items():
        lines += [
            f"### {workstream.replace('_', ' ').title()}",
            ""
        ]
        lines += [f"- {q}" for q in questions]
        lines.append("")

    lines += [
        "## New sources",
        ""
    ]
    lines += [
        f"- [[../14_Sources/{s['source_id']}]]"
        for s in local_sources
    ]

    lines += [
        "",
        "## New claims",
        ""
    ]
    lines += [
        f"- [[../15_Claims/{c['claim_id']}]] — "
        f"{c['confidence_score']}/100"
        for c in local_claims
    ]

    lines += [
        "",
        "## Contradiction refresh",
        ""
    ]
    lines += [
        f"- {c['contradiction_id']}: "
        f"**{c['refreshed_status']}** — "
        f"{c['resolution_note']}"
        for c in local_cons
    ] or ["- None"]

    lines += [
        "",
        "## Refreshed permission",
        "",
        f"**{permission['permission_state']}**",
        "",
        "Recommendation V1 remains preserved. No Recommendation V2 is created."
    ]

    write_note(
        diligence_dir / f"{mid}_Controlled_Diligence.md",
        lines
    )

print("Diligence notes:", len(list(diligence_dir.glob("*.md"))))


## Diligence readout

The readout explains:

- what was tested;
- what was learned;
- which contradictions were resolved or narrowed;
- how confidence changed;
- what remains uncertain;
- which next analytical capability is justified.


In [ ]:
readout = [
    "# Baby Step 5 — Controlled Diligence Readout",
    "",
    "## Executive conclusion",
    "",
    "Controlled synthetic diligence improved the evidentiary foundation of all five active matters.",
    "Several contradictions were resolved or narrowed, but Recommendation V1 remains the preserved baseline.",
    "",
    "## Portfolio results",
    "",
    f"- New diligence sources: **{len(new_sources)}**",
    f"- New diligence claims: **{len(new_claims)}**",
    f"- Resolved contradictions: **{status_counts.get('RESOLVED', 0)}**",
    f"- Narrowed contradictions: **{status_counts.get('NARROWED', 0)}**",
    f"- Remaining open contradictions: **{status_counts.get('REMAINS OPEN', 0)}**",
    "",
    "## Matter results",
    ""
]

for matter in matters:
    mid = matter["matter_id"]
    permission = next(
        p for p in refreshed_permissions
        if p["matter_id"] == mid
    )
    local_claims = [
        c for c in new_claims
        if c["matter_id"] == mid
    ]
    local_cons = [
        c for c in updated_contradictions
        if c["matter_id"] == mid
    ]

    readout += [
        f"### {mid} — {matter['caption']}",
        "",
        f"- New claims: {len(local_claims)}",
        f"- Average new-claim confidence: "
        f"{statistics.mean(c['confidence_score'] for c in local_claims):.2f}/100",
        f"- Refreshed total-claim confidence: "
        f"{permission['refreshed_average_claim_confidence']}/100",
        f"- Refreshed permission: **{permission['permission_state']}**",
        "- Contradiction status:"
    ]

    readout += [
        f"  - {c['contradiction_id']}: {c['refreshed_status']}"
        for c in local_cons
    ] or ["  - None"]

    readout += [
        ""
    ]

readout += [
    "## Governance conclusion",
    "",
    "The new evidence permits deeper internal strategy and remedies analysis.",
    "It does not authorize Recommendation V2 or external legal action."
]

write_note(
    VAULT / "10_Reports" / "Baby_Step_5_Controlled_Diligence_Readout.md",
    readout
)


## Human decision — DEC-005

DEC-005 accepts the refreshed evidence and permission state.

It authorizes:

- internal remedies and strategy design;
- damages and exposure sensitivity analysis;
- comparison of procedural options;
- continued contradiction resolution.

It still prohibits Recommendation V2 and external action.


In [ ]:
DECISION = {
    "decision_id": "DEC-005",
    "date": datetime.date.today().isoformat(),
    "title": "Accept Controlled Diligence and Refreshed Evidence",
    "decision": (
        "Accept the Baby Step 5 diligence findings, refreshed claims, "
        "contradiction states, and permission states as the new internal evidence baseline."
    ),
    "permitted_next_actions": [
        "internal remedies design",
        "damages and exposure sensitivity analysis",
        "procedural-option comparison",
        "continued contradiction resolution",
        "preserve Recommendation V1"
    ],
    "not_authorized": [
        "Recommendation V2",
        "filing",
        "service",
        "party contact",
        "court contact",
        "external counsel instruction",
        "settlement offer",
        "external legal advice",
        "external distribution"
    ],
    "synthetic": True
}

(VAULT / "09_Decisions" / "DEC-005.json").write_text(
    json.dumps(DECISION, indent=2),
    encoding="utf-8"
)

decision_lines = [
    "# DEC-005 — Accept Controlled Diligence and Refreshed Evidence",
    "",
    f"**Date:** {DECISION['date']}",
    "",
    "## Decision",
    "",
    DECISION["decision"],
    "",
    "## Permitted next actions",
    ""
]

decision_lines += [
    f"- {item}"
    for item in DECISION["permitted_next_actions"]
]

decision_lines += [
    "",
    "## Not authorized",
    ""
]

decision_lines += [
    f"- {item}"
    for item in DECISION["not_authorized"]
]

write_note(
    VAULT / "09_Decisions" / "DEC-005.md",
    decision_lines
)


## Hot-cache refresh

The hot cache now records the refreshed evidence state and the next permitted experiment: remedies, procedural options, damages, and strategy sensitivity.


In [ ]:
hot = [
    "# Current State — Hot Cache",
    "",
    "## Recommendation state",
    "",
    "- Recommendation V1 remains preserved.",
    "- Recommendation V2 does not exist.",
    "",
    "## Diligence state",
    "",
    f"- New diligence sources: {len(new_sources)}",
    f"- New diligence claims: {len(new_claims)}",
    f"- Resolved contradictions: {status_counts.get('RESOLVED', 0)}",
    f"- Narrowed contradictions: {status_counts.get('NARROWED', 0)}",
    f"- Remaining open contradictions: {status_counts.get('REMAINS OPEN', 0)}",
    "",
    "## Refreshed permission states",
    ""
]

hot += [
    f"- {p['matter_id']}: **{p['permission_state']}** — "
    f"confidence {p['refreshed_average_claim_confidence']}/100"
    for p in refreshed_permissions
]

hot += [
    "",
    "## Current decision",
    "",
    "- [[../09_Decisions/DEC-005]]",
    "",
    "## Permitted",
    "",
    "- Internal remedies design",
    "- Damages and exposure sensitivity analysis",
    "- Procedural-option comparison",
    "- Continued contradiction resolution",
    "",
    "## Prohibited",
    "",
    "- Recommendation V2",
    "- Filing or service",
    "- Party or court contact",
    "- Settlement offers",
    "- External legal advice",
    "",
    "## Next permitted experiment",
    "",
    "Design remedies, procedural options, damages scenarios, and strategy sensitivities."
]

write_note(
    VAULT / "12_Hot_Cache" / "Current_State.md",
    hot
)


## Validation

Baby Step 5 passes only if:

- five diligence records exist;
- ten new source records exist;
- ten new claim records exist;
- contradiction refresh states exist;
- refreshed permission states exist for all five matters;
- DEC-005 exists;
- Recommendation V1 remains preserved;
- Recommendation V2 does not exist.


In [ ]:
errors = []

diligence_notes = list(
    (VAULT / "17_Diligence").glob("*.md")
)

v1_notes = list(
    (VAULT / "08_Recommendations").glob("REC-*-V001.md")
)

v2_notes = list(
    (VAULT / "08_Recommendations").glob("REC-*-V002.md")
)

if len(diligence_notes) != 5:
    errors.append(
        f"Expected 5 diligence notes, found {len(diligence_notes)}"
    )

if len(new_sources) != 10:
    errors.append(
        f"Expected 10 new sources, found {len(new_sources)}"
    )

if len(new_claims) != 10:
    errors.append(
        f"Expected 10 new claims, found {len(new_claims)}"
    )

if len(refreshed_permissions) != 5:
    errors.append(
        "Refreshed permission states missing"
    )

if len(v1_notes) != 5:
    errors.append(
        f"Expected 5 Recommendation V1 notes, found {len(v1_notes)}"
    )

if v2_notes:
    errors.append(
        "Recommendation V2 exists prematurely"
    )

required = [
    VAULT / "10_Reports" / "Baby_Step_5_Controlled_Diligence_Readout.md",
    VAULT / "09_Decisions" / "DEC-005.md",
    VAULT / "09_Decisions" / "DEC-005.json",
    VAULT / "data" / "baby_step_5_diligence_plans.json",
    VAULT / "data" / "baby_step_5_refreshed_contradictions.json",
    VAULT / "data" / "baby_step_5_refreshed_permission_state.json"
]

for path in required:
    if not path.exists():
        errors.append(
            f"Missing required output: {path}"
        )

validation = {
    "validated_at": datetime.datetime.now().isoformat(),
    "diligence_note_count": len(diligence_notes),
    "new_source_count": len(new_sources),
    "new_claim_count": len(new_claims),
    "recommendation_v1_count": len(v1_notes),
    "recommendation_v2_count": len(v2_notes),
    "decision": "DEC-005",
    "errors": errors,
    "passed": len(errors) == 0
}

(VAULT / "11_Audit" / "Baby_Step_5_Validation.json").write_text(
    json.dumps(validation, indent=2),
    encoding="utf-8"
)

assert validation["passed"], errors

print(json.dumps(validation, indent=2))
print("BABY STEP 5 PASSED")


In [ ]:
state.update({
    "completed_steps": sorted(
        set(state.get("completed_steps", []) + [5])
    ),
    "current_step": 5,
    "next_step": 6,
    "decision": "DEC-005",
    "diligence_completed": True,
    "new_diligence_sources": len(new_sources),
    "new_diligence_claims": len(new_claims),
    "current_recommendation_version": 1,
    "next_problem": (
        "Design remedies, procedural options, damages scenarios, "
        "and strategy sensitivities."
    ),
    "permission_state": {
        "observe": True,
        "organize": True,
        "browse": True,
        "internal_strategy_analysis": True,
        "evidence_governance": True,
        "committee_product": True,
        "controlled_internal_diligence": True,
        "remedies_and_damages_analysis": True,
        "recommendation_v2": False,
        "external_action": False
    }
})

state_path.write_text(
    json.dumps(state, indent=2),
    encoding="utf-8"
)

audit = {
    "timestamp": datetime.datetime.now().isoformat(),
    "step": 5,
    "action": (
        "Executed controlled diligence, added new evidence, "
        "refreshed contradictions, and updated permission states."
    ),
    "outputs": {
        "new_sources": len(new_sources),
        "new_claims": len(new_claims),
        "diligence_records": 5,
        "decision": "DEC-005"
    },
    "validation_passed": validation["passed"]
}

with (VAULT / "11_Audit" / "workflow_audit.jsonl").open(
    "a",
    encoding="utf-8"
) as f:
    f.write(json.dumps(audit) + "\n")

print(json.dumps(state, indent=2))
